# Frontier RAG Benchmark (HotpotQA)

Grounded in HotpotQA (Yang et al., EMNLP 2018) — distractor setting.

**Cutting-edge angle:** not another single RAG flavor — a *system* that routes compute:

| Method | Role |
|--------|------|
| **FrontierRAG** | Classify → BM25+dense RRF → cross-encoder rerank → CRAG grade → escalate to hybrid |
| Adaptive / BM25+dense / Rerank | Ablations of Frontier building blocks |
| Semantic / Hybrid / GraphRAG modes | Classic baselines |
| HippoRAG 2 / LightRAG | Opt-in knowledge-graph RAG families |

**Reading scenario scores:** `0.357 \| 0.298` = same method on bridge vs comparison — not two mystery metrics.

See `results/routing_cheatsheet.md` and `choose_over_examples.csv` for data-driven choose-A-over-B.

In [ ]:
# ── Config ─────────────────────────────────────────────────
BENCHMARK = "hotpotqa"   # "hotpotqa" (paper) | "wikipedia" (custom)
N_HOTPOT = 12
FETCH_OR_BUILD = True
RUN_BENCHMARK = False    # results already in results/; set True to re-run
REUSE_INDEXES = True
METHODS = [
    "frontier_rag",          # Adaptive + CRAG escalate (cutting-edge system)
    "adaptive_rag",
    "hybrid_dense_sparse",
    "rerank_semantic",
    "semantic_rag",
    "hybrid_rag",
    "lazygraph_rag",         # GraphRAG fast/basic — not MS LazyGraphRAG
    # "hippo_rag", "light_rag",  # opt-in; slow OpenIE on 3B
]
# ───────────────────────────────────────────────────────────

import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from rag_benchmark import (
    BenchmarkConfig, BenchmarkRunner, create_tracked_client,
    fetch_corpus, build_hotpot_subset,
)
from rag_benchmark.charts import METHOD_LABELS, render_notebook_dashboard
from rag_benchmark.hybrid_rag import HybridRAG

# Guardrail: hybrid must not double-generate
src = (PROJECT_ROOT / "src/rag_benchmark/hybrid_rag.py").read_text()
assert "self.semantic.retrieve(" in src and "self.semantic.query(" not in src
print("Hybrid guardrail OK: retrieve-only + one fusion call")

config = BenchmarkConfig.from_yaml(PROJECT_ROOT)
config.project_root = PROJECT_ROOT
config.reuse_indexes = REUSE_INDEXES
print(f"Backend={config.llm_backend} chat={config.chat_model} embed={config.embedding_model}")

## 1. Build eval corpus

**HotpotQA distractor** builds a *closed* paragraph set from the paper JSON (no full-Wikipedia crawl/index).

In [ ]:
if BENCHMARK == "hotpotqa":
    if FETCH_OR_BUILD:
        paths = build_hotpot_subset(
            project_root=PROJECT_ROOT, n_questions=N_HOTPOT, prefer_hard=True
        )
        print(paths["meta"])
    config.corpus_dir = PROJECT_ROOT / "data" / "corpus_hotpot"
    config.qa_path = PROJECT_ROOT / "data" / "qa" / "hotpot_eval.json"
    # Separate Chroma collection + GraphRAG workspace so we don't mix wiki indexes
    config.semantic_collection = "hotpot_semantic"
    config.graph_workspace = PROJECT_ROOT / "graphrag_workspaces" / "hotpot"
    config.lazy_workspace = PROJECT_ROOT / "graphrag_workspaces" / "hotpot_lazy"
    config.max_documents = 10_000  # use all Hotpot paragraphs written for the subset
else:
    config.corpus_dir = PROJECT_ROOT / "data" / "corpus"
    config.qa_path = PROJECT_ROOT / "data" / "qa" / "eval_questions.json"
    if FETCH_OR_BUILD:
        titles = [
            "Albert Einstein", "Python (programming language)", "Machine learning",
            "Artificial intelligence", "Solar System", "Mars", "Jupiter",
            "World War II", "Neural network", "United States",
        ]
        fetch_corpus(titles, config.corpus_dir, max_chars=10_000)

n_docs = len(list(config.corpus_dir.glob("*.txt")))
print(f"Corpus: {config.corpus_dir} ({n_docs} docs)")
print(f"QA:     {config.qa_path}")

## 2. Run methods

GraphRAG uses **fast** indexing locally (spaCy NLP graph) — `standard` LLM extraction often fails on 3B models during community reports.

In [ ]:
results_dir = config.results_dir()

if RUN_BENCHMARK:
    runner = BenchmarkRunner(config, create_tracked_client(config))
    print(f"{len(runner.questions)} questions")
    for q in runner.questions[:5]:
        print(f"  [{q.query_type}] {q.question[:80]}")
    results = runner.run_all(methods=METHODS)
    saved = runner.save_results(results)
else:
    saved = {
        "summary_df": pd.read_csv(results_dir / "summary.csv"),
        "accuracy_df": pd.read_csv(results_dir / "accuracy_results.csv"),
        "latency_df": pd.read_csv(results_dir / "latency_results.csv"),
        "token_df": pd.read_csv(results_dir / "token_results.csv"),
        "scenario_df": pd.read_csv(results_dir / "scenario_results.csv")
        if (results_dir / "scenario_results.csv").exists() else pd.DataFrame(),
    }
    results = None
    print("Loaded saved results")

## Key finding (HotpotQA distractor, n=12)

| Scenario | Winner | Why |
|----------|--------|-----|
| **local** (comparison / single-fact) | **Semantic** (0.52) | Phrase match in one paragraph |
| **hybrid** (bridge / multi-hop) | **Hybrid / fast-basic** (~0.35) | Needs both docs; vector-only drops to 0.30; GraphRAG local alone fails (0.03) |

### Decoding “fast/basic … 0.357 / 0.298”

Same method, **two Hotpot scenarios** (mean composite quality):

| Number | Scenario | Meaning |
|--------|----------|---------|
| **0.357** | `hybrid` column | Hotpot **bridge** (multi-hop) questions |
| **0.298** | `local` column | Hotpot **comparison** questions |

So GraphRAG fast/basic is decent on multi-hop for a cheap NLP index, but loses to vector RAG on simple comparisons. It is **not** LazyGraphRAG (that full system still isn’t in the open-source GraphRAG CLI).

Overall averages look semantic-heavy because half the set is local comparison.

## When to choose which RAG (from *your* data)

Cutting-edge production pattern is **Adaptive-RAG routing** — not crowning one stack. Below is generated from `results/accuracy_results.csv` (Hotpot n=12).



In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
from rag_benchmark.decision_playbook import build_decision_artifacts
from rag_benchmark.examples_playbook import format_examples_markdown

qa_path = Path(config.qa_path) if "config" in dir() else ROOT / "data" / "qa" / "hotpot_eval.json"
results_dir = Path(RESULTS_DIR) if "RESULTS_DIR" in dir() else ROOT / "results"

paths = build_decision_artifacts(results_dir, qa_path)
display(Markdown(Path(paths["cheatsheet"]).read_text(encoding="utf-8")))
display(Markdown("### Choose-A-over-B table"))
display(__import__("pandas").read_csv(paths["examples"]))
display(Markdown("### Pairwise win rates (BenchmarkQED-lite)"))
display(__import__("pandas").read_csv(paths["pairwise"]).head(12))
display(Markdown("---\n### Method intuition cards"))
display(Markdown(format_examples_markdown()))


## Engineering scorecard

What teams actually need: **routing** (which method for which query), **p95 latency**, **tokens/query**, and a default production pick — not just a single overall average.

In [ ]:
from IPython.display import Image, Markdown, display
from rag_benchmark.engineering import (
    build_engineering_scorecard,
    print_engineering_briefing,
    save_engineering_scorecard,
)

if "summary_df" not in dir() or summary_df is None:
    summary_df = pd.read_csv(RESULTS_DIR / "summary.csv")
    accuracy_df = pd.read_csv(RESULTS_DIR / "accuracy_results.csv")
    scenario_df = (
        pd.read_csv(RESULTS_DIR / "scenario_results.csv")
        if (RESULTS_DIR / "scenario_results.csv").exists()
        else pd.DataFrame()
    )

scorecard = build_engineering_scorecard(
    summary_df=summary_df,
    scenario_df=scenario_df,
    accuracy_df=accuracy_df,
)
eng_paths = save_engineering_scorecard(scorecard, RESULTS_DIR)
print_engineering_briefing(scorecard)

display(Markdown("### Method scorecard"))
display(pd.DataFrame(scorecard["methods"])[
    [
        c
        for c in [
            "label",
            "mean_composite_score",
            "usable_answer_rate",
            "tokens_per_query",
            "quality_per_1k_tokens",
            "p95_query_latency_seconds",
            "meets_latency_slo",
        ]
        if c in pd.DataFrame(scorecard["methods"]).columns
    ]
])
display(Markdown("### Routing recommendations"))
display(pd.DataFrame(scorecard["routing_by_query_type"]))
if Path(eng_paths["chart"]).exists():
    display(Image(filename=str(eng_paths["chart"])))
display(Markdown(Path(eng_paths["briefing"]).read_text(encoding="utf-8")))


## 3. Charts + leaderboard (quality, tokens, scenarios)

In [ ]:
frames = render_notebook_dashboard(saved=saved, results_dir=results_dir)
summary = frames["summary"]
cols = [c for c in [
    "label", "mean_composite_score", "mean_llm_judge", "mean_token_f1",
    "exact_match_rate", "total_tokens", "tokens_per_query",
    "mean_query_latency_seconds",
] if c in summary.columns]
display(summary.sort_values("mean_composite_score", ascending=False)[cols].round(3))

In [ ]:
scenario = frames.get("scenario")
if scenario is not None and not scenario.empty:
    pivot = scenario.pivot_table(
        index="label", columns="query_type", values="mean_composite_score", observed=False
    ).round(3)
    print("Quality by Hotpot-style scenario (bridge→hybrid, comparison→local):")
    display(pivot)

print("\nToken efficiency (quality per 1k tokens):")
eff = summary.copy()
eff["quality_per_1k_tokens"] = (
    eff["mean_composite_score"] / (eff["total_tokens"].clip(lower=1) / 1000)
).round(3)
display(eff[["label", "mean_composite_score", "total_tokens", "quality_per_1k_tokens"]]
        .sort_values("quality_per_1k_tokens", ascending=False))

## 4. Sample answers

In [ ]:
if results:
    for result in results:
        print(f"\n=== {METHOD_LABELS.get(result.method, result.method)} | tokens={result.ledger.total().total_tokens} ===")
        for row in result.answers[:2]:
            print(f"[{row.get('query_type')}] Q: {row['question']}")
            print(f"A: {row['answer'][:350]}\n")
else:
    print("Set RUN_BENCHMARK=True for live answers.")